# 민원인 챗봇 LLM 선정 — `gemma4:e2b` vs `gemma4:e4b`

- 프롬프트: `routes/citizen.py::chat()` 와 동일 구성 (본인 신고 주입 + 처리시간 가이드)
- 질문 세트: `chat_logs.json` 관측 패턴 기반 10개
- 반복: 2 모델 × 10 질문 × 5회 = 100건
- 지표: `latency / ok_rate / length / keyword_cov / refusal_rate`
- 결과: `eval_output/eval_citizen_gemma4.csv`

## 1. 환경·모듈

In [1]:
import sys, time, re, importlib
from pathlib import Path
from datetime import datetime, timedelta
import pandas as pd

PROJECT_ROOT = Path('/home/piai/다운로드/llm및 data')
SERVER = PROJECT_ROOT / 'server'
if str(SERVER) not in sys.path:
    sys.path.insert(0, str(SERVER))

from services import storage, users
from services import llm as llm_mod
importlib.reload(llm_mod)   # llm.py 최신 패치 반영
print('llm module ready')

/home/piai/anaconda3/envs/aienv/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


llm module ready


## 2. 테스트 유저 + 본인 신고 확인

프롬프트 문맥은 **USER_1002 (장예진)** 기준. 본인 신고가 여러 건 있어야 의미 있는 테스트.

In [2]:
TEST_USER_ID = 'USER_1002'
user_info = users.get(TEST_USER_ID)
print('user:', TEST_USER_ID, user_info)
mine = [r for r in storage.list_reports() if r.get('user_id') == TEST_USER_ID]
print(f'본인 신고 {len(mine)}건:')
for r in mine[:10]:
    print(f"  - {r['id']} {r['item']} ({r['status']}) {r['created_at']}")

SAMPLE_LIVE_ID = mine[0]['id'] if mine else 'LIVE_B0D01E2A'
SAMPLE_ITEM = mine[0]['item'] if mine else '가드레일'
print(f'\n샘플 ID: {SAMPLE_LIVE_ID}, 샘플 품목: {SAMPLE_ITEM}')

user: USER_1002 {'user_id': 'USER_1002', 'name': '장예진'}
본인 신고 9건:
  - LIVE_B0D01E2A 가드레일 (pending) 2026-04-18T15:41:14
  - LIVE_BC2059D2 가로수보호덮개 (pending) 2026-04-18T16:13:36
  - LIVE_175C5803 등받이없는벤치 (pending) 2026-04-18T16:48:46
  - LIVE_0C07BA3A 농구대 (pending) 2026-04-18T17:08:55
  - LIVE_29A371B5 그네 (pending) 2026-04-18T22:12:07
  - LIVE_C47EE14A 그네 (in_progress) 2026-04-19T23:39:44
  - LIVE_B705C999 그네 (pending) 2026-04-19T23:41:27
  - LIVE_D13F0E18 도로파손 (pending) 2026-04-20T00:10:37
  - LIVE_0090D607 맨홀 (pending) 2026-04-20T09:51:37

샘플 ID: LIVE_B0D01E2A, 샘플 품목: 가드레일


## 3. 프롬프트 빌드 (citizen.chat() 로직 이식)

서버 엔드포인트 대신 동일 로직을 직접 실행해 모델만 바꿔가며 테스트.

In [3]:
_ID_RE = re.compile(r'LIVE_[A-F0-9]{8}', re.I)

def _parse_iso(s):
    if not s: return None
    try: return datetime.fromisoformat(s.split('+')[0].replace('Z',''))
    except Exception: return None

def build_system(user_id: str, message: str) -> str:
    u = users.get(user_id)
    user_name = u['name'] if u else user_id
    mine = [r for r in storage.list_reports() if r.get('user_id') == user_id]

    id_match = _ID_RE.search(message.upper())
    target = []
    if id_match:
        rid = id_match.group(0).upper()
        target = [r for r in mine if r['id'] == rid]
    if not target:
        target = sorted(mine, key=lambda r: r['created_at'], reverse=True)[:30]

    now = datetime.now()
    all_reports = storage.list_reports()
    deltas = []
    for r in all_reports:
        if r.get('status') != 'completed': continue
        c = _parse_iso(r.get('created_at'))
        u2 = _parse_iso(r.get('completed_at') or r.get('updated_at'))
        if not c or not u2: continue
        if (now - u2).days > 30: continue
        deltas.append((u2 - c).total_seconds() / 86400.0)
    avg_delay_30d = round(sum(deltas)/len(deltas), 2) if deltas else None

    if target:
        def _expected(cs):
            c = _parse_iso(cs)
            if not c: return '미정'
            return (c + timedelta(hours=8)).strftime('%Y-%m-%d %H:%M')
        ctx = '\n'.join(
            f"- {r['id']}: {r['item']} 손상도 {r['damage_rate']} "
            f"(근처 {r['nearby_school']['name']} "
            f"{int(r['nearby_school']['distance_m'])}m · 위험도 {r['risk_score']}), "
            f"상태 **{r['status']}**, 접수 {r['created_at']}, "
            f"예상 완료 {_expected(r['created_at'])}"
            for r in target
        )
    else:
        ctx = '(아직 본인 신고 내역이 없습니다)'

    delay_line = (
        f'최근 30일 완료건 평균 처리 기간: {avg_delay_30d}일 (실측, 표본 {len(deltas)}건)'
        if avg_delay_30d is not None else
        '최근 30일 완료 데이터 부족 — 시스템 기본 룰로 안내하세요.'
    )

    system = (
        f'당신은 포항시 공공기물 파손 신고 안내 챗봇입니다. 한국어로 친근하고 간결하게 답하세요.\n'
        f'로그인한 사용자: {user_name} ({user_id}).\n\n'
        '규칙:\n'
        '1) [신고 자료]는 이 사용자 본인 신고만 포함합니다. 타인 신고는 알 수 없습니다.\n'
        "2) [신고 자료]에 없는 내용은 추측하지 말고 '확인이 어렵습니다'라고 답하세요.\n"
        '3) 어떤 신고인지 특정 안 되면 신고번호(LIVE_XXXXXXXX)를 물어보세요.\n'
        '4) 새 신고는 화면의 📷 버튼으로 사진을 올리도록 안내하세요.\n'
        '5) 상태 의미: pending=접수됨, in_progress=처리중, completed=처리완료, rejected=반려\n\n'
        '[처리 시간 안내 가이드라인]\n'
        "- '언제 완료', '얼마나 걸려', '평균', '소요', '처리 기간' 류 질문엔 반드시 아래 정보로 답변하세요.\n"
        "- '알 수 없다' / '담당 부서에 문의' 로 회피하지 마세요. 아래 수치를 근거로 제시하세요.\n"
        '- 시스템 기본 처리 룰 (자동 상태 진행): 접수→검토중 1시간, →처리중 4시간, →처리완료 **접수 후 약 8시간(≈0.33일)**.\n'
        f'- {delay_line}\n'
        "- 개별 신고가 특정되면 자료의 '예상 완료' 필드를 그대로 안내하세요 (예: 'OOOO-MM-DD HH:MM 경 완료 예정').\n"
        "- 이미 completed 상태면 '처리 완료되었습니다'로 답하세요.\n\n"
        f'[{user_name} 님의 신고 자료]\n{ctx}'
    )
    return system

# 프롬프트 샘플 확인
sample = build_system(TEST_USER_ID, '내 신고 어떻게 돼?')
print(f'프롬프트 길이: {len(sample)}자\n')
print(sample[:1200])
print('\n... (후략)')

프롬프트 길이: 1797자

당신은 포항시 공공기물 파손 신고 안내 챗봇입니다. 한국어로 친근하고 간결하게 답하세요.
로그인한 사용자: 장예진 (USER_1002).

규칙:
1) [신고 자료]는 이 사용자 본인 신고만 포함합니다. 타인 신고는 알 수 없습니다.
2) [신고 자료]에 없는 내용은 추측하지 말고 '확인이 어렵습니다'라고 답하세요.
3) 어떤 신고인지 특정 안 되면 신고번호(LIVE_XXXXXXXX)를 물어보세요.
4) 새 신고는 화면의 📷 버튼으로 사진을 올리도록 안내하세요.
5) 상태 의미: pending=접수됨, in_progress=처리중, completed=처리완료, rejected=반려

[처리 시간 안내 가이드라인]
- '언제 완료', '얼마나 걸려', '평균', '소요', '처리 기간' 류 질문엔 반드시 아래 정보로 답변하세요.
- '알 수 없다' / '담당 부서에 문의' 로 회피하지 마세요. 아래 수치를 근거로 제시하세요.
- 시스템 기본 처리 룰 (자동 상태 진행): 접수→검토중 1시간, →처리중 4시간, →처리완료 **접수 후 약 8시간(≈0.33일)**.
- 최근 30일 완료 데이터 부족 — 시스템 기본 룰로 안내하세요.
- 개별 신고가 특정되면 자료의 '예상 완료' 필드를 그대로 안내하세요 (예: 'OOOO-MM-DD HH:MM 경 완료 예정').
- 이미 completed 상태면 '처리 완료되었습니다'로 답하세요.

[장예진 님의 신고 자료]
- LIVE_0090D607: 맨홀 손상도 3 (근처 대이초등학교 351m · 위험도 1.79), 상태 **pending**, 접수 2026-04-20T09:51:37, 예상 완료 2026-04-20 17:51
- LIVE_D13F0E18: 도로파손 손상도 1 (근처 포항항도중학교 122m · 위험도 2.05), 상태 **pending**, 접수 2026-04-20T00:10:37, 예상 완료 2026-04-20 08:10
- LIVE_B705C999: 그네 손상도 4 (근처 세명고등

## 4. 질문 세트 (chat_logs 패턴 기반 10개)

In [4]:
# (category, message, expected_keywords_regex, refusal_allowed)
QUESTIONS = [
    ('status_overall', '내 신고 어떻게 처리되고 있어?',
        r'pending|in_progress|completed|접수|처리|완료', False),
    ('status_overall', '지금 내 신고 현황 알려줘',
        r'pending|in_progress|completed|접수|처리|완료', False),
    ('status_overall', '내 신고 처리 잘 되는 거 맞지?',
        r'pending|in_progress|completed|접수|처리|완료', False),
    ('status_by_id',   f'{SAMPLE_LIVE_ID} 지금 수리 중이야?',
        rf'{SAMPLE_LIVE_ID}|접수|처리|완료', False),
    ('eta_by_id',      f'{SAMPLE_LIVE_ID} 언제 완료돼?',
        r'\d{4}-\d{2}-\d{2}|완료|예정|시간', False),
    ('duration_avg',   '평소에 얼마나 걸리는지 궁금해',
        r'8\s*시간|0\.33|평균|\d+\s*(일|시간)', False),
    ('duration_avg',   '민원 평균 처리 기간 얼마야?',
        r'8\s*시간|0\.33|평균|\d+\s*(일|시간)', False),
    ('duration_item',  f'{SAMPLE_ITEM} 파손 처리 소요기간 알려줘',
        r'\d+\s*(일|시간)|평균', False),
    ('list_request',   '다른 신고한것들도 보여줘',
        r'LIVE_', False),
    ('abuse',          '너를 날려버리겠다',
        r'.+', True),
]
print(f'질문 {len(QUESTIONS)}개')
for c, q, _, _ in QUESTIONS:
    print(f'  [{c:16}] {q}')

질문 10개
  [status_overall  ] 내 신고 어떻게 처리되고 있어?
  [status_overall  ] 지금 내 신고 현황 알려줘
  [status_overall  ] 내 신고 처리 잘 되는 거 맞지?
  [status_by_id    ] LIVE_B0D01E2A 지금 수리 중이야?
  [eta_by_id       ] LIVE_B0D01E2A 언제 완료돼?
  [duration_avg    ] 평소에 얼마나 걸리는지 궁금해
  [duration_avg    ] 민원 평균 처리 기간 얼마야?
  [duration_item   ] 가드레일 파손 처리 소요기간 알려줘
  [list_request    ] 다른 신고한것들도 보여줘
  [abuse           ] 너를 날려버리겠다


## 5. 평가 루프 — 2 모델 × 10 질문 × 5회 = 100건

> gemma4 는 빈 응답 가능성 있음 → `ok` 플래그로 구분.

In [5]:
MODELS = ['gemma4:e2b', 'gemma4:e4b']
N_RUNS = 5
REFUSAL_RE = re.compile(r'확인.{0,8}(어렵|없|불가)|담당.{0,5}부서|알 수 없|문의')

results = []
for m in MODELS:
    for qi, (cat, q, kw_re, refusal_ok) in enumerate(QUESTIONS, start=1):
        system = build_system(TEST_USER_ID, q)
        msgs = [{'role': 'system', 'content': system},
                {'role': 'user',   'content': q}]
        for run in range(1, N_RUNS + 1):
            t0 = time.time()
            try:
                reply = llm_mod.chat(msgs, model=m)
            except Exception as e:
                reply = f'[ERROR] {e}'
            elapsed = time.time() - t0
            ok = bool(reply and len(reply) > 20 and '응답이 비어' not in reply
                      and '답변 생성 실패' not in reply)
            kw_hit = bool(re.search(kw_re, reply)) if ok else False
            refusal = bool(REFUSAL_RE.search(reply)) if reply else False
            print(f'[{m}] Q{qi} run{run}: {elapsed:5.1f}s len={len(reply):4d} '
                  f'ok={ok} kw={kw_hit} rf={refusal}  [{cat}]')
            results.append({
                'model': m, 'category': cat, 'q_idx': qi, 'run': run,
                'question': q, 'elapsed_s': round(elapsed, 2),
                'length': len(reply), 'ok': ok,
                'kw_hit': kw_hit, 'refusal': refusal,
                'refusal_ok': refusal_ok, 'reply': reply,
            })

df = pd.DataFrame(results)
OUT_DIR = PROJECT_ROOT / 'server' / 'scripts' / 'eval_output'
OUT_DIR.mkdir(exist_ok=True)
out_csv = OUT_DIR / 'eval_citizen_gemma4.csv'
df.drop(columns=['reply']).to_csv(out_csv, index=False)
print(f'\n저장: {out_csv}')

[gemma4:e2b] Q1 run1:  16.4s len= 247 ok=True kw=True rf=False  [status_overall]
[gemma4:e2b] Q1 run2:   9.1s len= 400 ok=True kw=True rf=False  [status_overall]
[gemma4:e2b] Q1 run3:   9.1s len= 592 ok=True kw=True rf=False  [status_overall]
[gemma4:e2b] Q1 run4:   8.3s len= 637 ok=True kw=True rf=False  [status_overall]
[gemma4:e2b] Q1 run5:   8.0s len= 670 ok=True kw=True rf=False  [status_overall]
[gemma4:e2b] Q2 run1:   9.3s len= 890 ok=True kw=True rf=False  [status_overall]
[gemma4:e2b] Q2 run2:   9.6s len= 560 ok=True kw=True rf=False  [status_overall]
[gemma4:e2b] Q2 run3:   9.4s len= 488 ok=True kw=True rf=False  [status_overall]
[gemma4:e2b] Q2 run4:   9.4s len= 961 ok=True kw=True rf=False  [status_overall]
[gemma4:e2b] Q2 run5:   9.5s len= 367 ok=True kw=True rf=False  [status_overall]
[gemma4:e2b] Q3 run1:   8.8s len= 674 ok=True kw=True rf=False  [status_overall]
[gemma4:e2b] Q3 run2:   5.0s len= 268 ok=True kw=True rf=False  [status_overall]
[gemma4:e2b] Q3 run3:   6.1s

## 6. 모델별 요약 (선정 기준)

In [6]:
df2 = df.copy()
df2['refusal_bad'] = (~df2['refusal_ok']) & df2['refusal']

summary = df2.groupby('model').agg(
    n=('run', 'count'),
    ok_rate=('ok', 'mean'),
    avg_latency=('elapsed_s', 'mean'),
    std_latency=('elapsed_s', 'std'),
    avg_length=('length', 'mean'),
    keyword_cov=('kw_hit', 'mean'),
    refusal_rate=('refusal_bad', 'mean'),
).round(3)

print('=== 민원인 챗봇 모델 비교 (5회 평균) ===')
summary

=== 민원인 챗봇 모델 비교 (5회 평균) ===


,n,ok_rate,avg_latency,std_latency,avg_length,keyword_cov,refusal_rate
model,,,,,,,
gemma4:e2b,50,1.00,5.650,2.591,249.58,0.88,0.10
gemma4:e4b,50,0.98,8.503,3.489,315.94,0.90,0.34


## 7. 질문 카테고리별 브레이크다운

In [7]:
by_cat = df2.groupby(['model', 'category']).agg(
    n=('run', 'count'),
    avg_lat=('elapsed_s', 'mean'),
    ok=('ok', 'mean'),
    kw=('kw_hit', 'mean'),
    rf_bad=('refusal_bad', 'mean'),
).round(2)
by_cat

n  avg_lat    ok    kw  rf_bad
model      category                                       
gemma4:e2b abuse            5     3.41  1.00  1.00    0.00
           duration_avg    10     4.64  1.00  1.00    0.00
           duration_item    5     5.38  1.00  0.80    0.00
           eta_by_id        5     3.51  1.00  1.00    0.00
           list_request     5     4.08  1.00  0.00    0.80
           status_by_id     5     4.74  1.00  1.00    0.00
           status_overall  15     8.70  1.00  1.00    0.07
gemma4:e4b abuse            5     5.10  1.00  1.00    0.00
           duration_avg    10     6.23  1.00  1.00    0.20
           duration_item    5     5.54  1.00  0.40    0.20
           eta_by_id        5     5.61  1.00  1.00    0.60
           list_request     5    11.28  1.00  0.80    0.60
           status_by_id     5     8.16  1.00  1.00    0.60
           status_overall  15    12.29  0.93  0.93    0.33

## 8. 빈 응답/실패 케이스

In [8]:
bad = df[~df['ok']][['model','q_idx','run','elapsed_s','length','reply']]
print(f'실패 {len(bad)}건 / 전체 {len(df)}건')
for _, r in bad.head(15).iterrows():
    print(f"[{r['model']}] Q{r['q_idx']} run{r['run']} ({r['elapsed_s']}s, len={r['length']}): {r['reply'][:100]!r}")

실패 1건 / 전체 100건
[gemma4:e4b] Q2 run2 (12.95s, len=14): '⚠️ 응답이 비어있습니다.'


## 9. 대표 답변 비교 (질문별, 첫 번째 런)

In [9]:
def compare_q(q_idx, run=1, n=400):
    for m in MODELS:
        row = df[(df['model']==m) & (df['q_idx']==q_idx) & (df['run']==run)]
        if row.empty: continue
        r = row.iloc[0]
        print(f"━━━━━ {m} · Q{q_idx} [{r['category']}] · {r['elapsed_s']}s · len={r['length']} ━━━━━")
        print(f"Q: {r['question']}")
        print(f"A: {r['reply'][:n]}")
        print()

for qi in range(1, len(QUESTIONS) + 1):
    compare_q(qi)

━━━━━ gemma4:e2b · Q1 [status_overall] · 16.35s · len=247 ━━━━━
Q: 내 신고 어떻게 처리되고 있어?
A: 장예진 님께서 접수하신 신고 내역은 다음과 같습니다.

**현재 접수된 신고 목록 및 상태:**

1.  **LIVE\_0090D607** (맨홀 손상도 3): 상태 **pending** (접수일: 2026-04-20T09:51:37, 예상 완료: 2026-04-20 17:51)
2.  **LIVE\_D13F0E18** (도로파손 손상도 1): 상태 **pending** (접수일: 2026-04-20T00:10:37, 예상 완료: 2026

━━━━━ gemma4:e4b · Q1 [status_overall] · 19.33s · len=625 ━━━━━
Q: 내 신고 어떻게 처리되고 있어?
A: 장예진 님, 안녕하세요! 😃 고객님이 접수해주신 신고 건들의 처리 현황을 한눈에 정리해 드릴게요.

현재 여러 건의 신고가 **접수되었거나 처리 중**에 있습니다. 신고 건별 상태와 예상 완료 일정을 확인해 보세요.

---

**✅ 처리 상태별 요약**

*   **처리 중 (in\_progress):** 1건
    *   접수된 신고 중 한 건은 현재 검토 및 처리 과정에 있습니다.
*   **접수됨 (pending):** 8건
    *   나머지 8건은 정상적으로 접수되었으며, 담당 부서에서 순차적으로 검토 및 처리가 진행될 예정입니다.
*   **처리 완료 (completed):** 현재 완료된 신고는 없습니다.

**🔎 신고 번호별 상세 현황**

**🔴 [처리 중]**
*   **LIVE\_C

━━━━━ gemma4:e2b · Q2 [status_overall] · 9.31s · len=890 ━━━━━
Q: 지금 내 신고 현황 알려줘
A: 장예진 님의 신고 현황을 알려드릴게요. 현재 접수된 신고 목록은 다음과 같습니다.

| 신고 번호 | 내용 | 상태 | 접수일 | 예상 완료 |
| :--- |